# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [3]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


Python 실행 파일: c:\dev\llm-data-analysis-study\.venv\Scripts\python.exe
현재 작업 폴더: g:\내 드라이브\dev_GD\llm-data-analysis-study\notebooks
프로젝트 루트: G:\내 드라이브\dev_GD\llm-data-analysis-study
데이터 폴더: G:\내 드라이브\dev_GD\llm-data-analysis-study\data\raw
결과 저장 폴더: G:\내 드라이브\dev_GD\llm-data-analysis-study\reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [4]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [5]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [6]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


## 5. 정렬과 주문 상태 확인


In [7]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-06-04
8,9,송지민,M,69,서울,2025-10-19
54,55,윤영철,M,69,광주,2023-07-31
78,79,이서준,M,69,부산,2023-07-14
136,137,김선영,M,68,울산,2024-11-28
45,46,김서현,M,67,대전,2026-05-13
132,133,김성진,M,67,성남,2026-05-29
123,124,김시우,M,67,대구,2024-09-24
11,12,김정남,F,66,대전,2024-09-30
57,58,김성훈,F,66,성남,2025-06-14


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [8]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [9]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [10]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


## 9. 상품 데이터 병합과 검증


In [11]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 10. 카테고리별·상품별 매출


In [12]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


## 11. 월별 매출


In [13]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-07,5869000,8,733625.0
1,2025-08,15621000,18,867833.0
2,2025-09,10190000,13,783846.0
3,2025-10,25766000,26,991000.0
4,2025-11,8812000,12,734333.0
5,2025-12,11501000,14,821500.0
6,2026-01,17423000,22,791955.0
7,2026-02,9749000,17,573471.0
8,2026-03,13429000,14,959214.0
9,2026-04,17553000,23,763174.0


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [14]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


## 13. 결과 저장하기


In [15]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)


ch04_category_sales.csv True 254
ch04_product_sales.csv True 4409
ch04_monthly_sales.csv True 442
ch04_customer_sales.csv True 3387


In [16]:
base_total = completed_order_sales['line_total'].sum()
category_total = category_sales['total_sales'].sum()
product_total = product_sales['total_sales'].sum()
monthly_total = monthly_summary['total_sales'].sum()
customer_total = customer_sales['total_sales'].sum()

print('원본 completed 합계:', base_total)
print('카테고리 합계:', category_total)
print('상품 합계:', product_total)
print('월별 합계:', monthly_total)
print('고객별 합계:', customer_total)

print(
    '전체 합계 일치:',
    base_total
    == category_total
    == product_total
    == monthly_total
    == customer_total
)

원본 completed 합계: 148990000
카테고리 합계: 148990000
상품 합계: 148990000
월별 합계: 148990000
고객별 합계: 148990000
전체 합계 일치: True


## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```


제안된 코드는 현재 데이터에서 그대로 사용할 수 없다.
category 컬럼은 order_items가 아니라 products에 있기 때문에
현재 코드에서는 category를 기준으로 바로 집계할 수 없다.
따라서 이번 과제에서 진행한 대로 아래순서로 검증을 진행해야 된다.
1. order_items.order_id와 orders.order_id를 연결한다.
2. validate="many_to_one"과 indicator=True를 사용해 병합을 검증한다.
3. 병합 전후 행 수와 미매칭 여부를 확인한다.
4. order_status == "completed"인 주문만 선택한다.
5. product_id를 기준으로 products를 연결한다.
6. 상품 병합에서도 validate="many_to_one"과 indicator=True로 검증한다.
7. category별로 line_total을 합산한다.
8. 최종 카테고리 합계를 원본 completed line_total 합계와 비교한다.

In [20]:
# 1. 실제 컬럼 위치 확인
print("order_items 컬럼:", order_items.columns.tolist())
print("orders 컬럼:", orders.columns.tolist())
print("products 컬럼:", products.columns.tolist())

print("\ncategory 위치 확인:",
      "category" in products.columns)
print("order_status 위치 확인:",
      "order_status" in orders.columns)


# 2. orders와 병합하기 전 key 고유성 확인
print("\norders.order_id 중복 수:",
      orders["order_id"].duplicated().sum())


# 3. order_items + orders 병합
order_sales_check = order_items.merge(
    orders[
        [
            "order_id",
            "customer_id",
            "order_date",
            "order_status",
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator="order_match",
)

print("\n[order_items + orders 병합 검증]")
print("병합 전 행 수:", len(order_items))
print("병합 후 행 수:", len(order_sales_check))
display(
    order_sales_check["order_match"]
    .value_counts(dropna=False)
)


# 4. completed 주문만 선택
completed_sales_check = order_sales_check[
    order_sales_check["order_status"] == "completed"
].copy()

print("\n[completed 필터 확인]")
display(
    completed_sales_check["order_status"]
    .value_counts(dropna=False)
)

print(
    "completed 주문 상세 금액 합계:",
    completed_sales_check["line_total"].sum()
)


# 5. products와 병합하기 전 key 고유성 확인
print(
    "\nproducts.product_id 중복 수:",
    products["product_id"].duplicated().sum()
)


# 6. completed 주문 상세 + products 병합
completed_items_check = completed_sales_check.merge(
    products[
        [
            "product_id",
            "product_name",
            "category",
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="product_match",
)

print("\n[completed 주문 + products 병합 검증]")
print("병합 전 행 수:", len(completed_sales_check))
print("병합 후 행 수:", len(completed_items_check))
display(
    completed_items_check["product_match"]
    .value_counts(dropna=False)
)


# 7. 카테고리별 completed 주문 금액 집계
category_sales_check = (
    completed_items_check
    .groupby("category", as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)

print("\n[카테고리별 completed 주문 기준 금액]")
display(category_sales_check)


# 8. 원본 completed 합계와 카테고리 합계 검증
completed_total = completed_sales_check["line_total"].sum()
category_total = category_sales_check["total_sales"].sum()

print("\n[최종 합계 검증]")
print("원본 completed 합계:", completed_total)
print("카테고리 합계:", category_total)
print("합계 일치:", completed_total == category_total)

order_items 컬럼: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']
orders 컬럼: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
products 컬럼: ['product_id', 'product_name', 'category', 'price']

category 위치 확인: True
order_status 위치 확인: True

orders.order_id 중복 수: 0

[order_items + orders 병합 검증]
병합 전 행 수: 764
병합 후 행 수: 764


order_match
both          764
left_only       0
right_only      0
Name: count, dtype: int64


[completed 필터 확인]


order_status
completed    474
Name: count, dtype: int64

completed 주문 상세 금액 합계: 148990000

products.product_id 중복 수: 0

[completed 주문 + products 병합 검증]
병합 전 행 수: 474
병합 후 행 수: 474


product_match
both          474
left_only       0
right_only      0
Name: count, dtype: int64


[카테고리별 completed 주문 기준 금액]


,category,total_quantity,total_sales
3,스포츠,295,31743000
5,전자기기,259,26400000
2,생활용품,272,23915000
1,뷰티,223,23383000
4,식품,133,16573000
0,도서,149,16389000
6,패션,111,10587000



[최종 합계 검증]
원본 completed 합계: 148990000
카테고리 합계: 148990000
합계 일치: True


## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [21]:
# 과제 코드를 아래에 작성하세요.
# 1. 40세 이상 고객 추출

customers_over_40 = customers[
    customers["age"] >= 40
]

print("40세 이상 고객 수:", len(customers_over_40))
display(customers_over_40.head())

# 2. 상품 가격이 낮은 순서대로 10개 출력

low_price_products = (
    products
    .sort_values("price", ascending=True)
    .head(10)
)

display(low_price_products)


# 3. 결제수단별 주문 수와 비율 계산

payment_count = orders["payment_method"].value_counts()
payment_ratio = (
    orders["payment_method"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

payment_summary = pd.DataFrame({
    "order_count": payment_count,
    "ratio_percent": payment_ratio,
})

display(payment_summary)

# 4. 카테고리별 평균 상품 가격 계산

category_average_price = (
    products
    .groupby("category", as_index=False)
    .agg(
        average_price=("price", "mean")
    )
)

display(category_average_price)


# 5. 완료 주문과 전체 주문의 금액 차이 계산

total_amount = order_items["line_total"].sum()
completed_amount = completed_order_sales["line_total"].sum()
amount_difference = total_amount - completed_amount

print("전체 주문 상세 금액:", total_amount)
print("completed 주문 기준 금액:", completed_amount)
print("금액 차이:", amount_difference)



40세 이상 고객 수: 78


,customer_id,name,gender,age,city,signup_date
2,3,이경수,F,61,성남,2024-06-12
3,4,조영호,F,55,울산,2026-04-13
6,7,이상현,F,53,인천,2024-12-12
8,9,송지민,M,69,서울,2025-10-19
9,10,유도현,F,62,울산,2024-10-06


,product_id,product_name,category,price
5,6,전자기기 상품 006,전자기기,5000
27,28,뷰티 상품 028,뷰티,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000


,order_count,ratio_percent
payment_method,,
kakao_pay,79,26.33
naver_pay,77,25.67
bank_transfer,74,24.67
card,70,23.33


,category,average_price
0,도서,106857.142857
1,뷰티,117687.500000
2,생활용품,96437.500000
3,스포츠,111578.947368
4,식품,137142.857143
5,전자기기,101588.235294
6,패션,115909.090909


전체 주문 상세 금액: 255610000
completed 주문 기준 금액: 148990000
금액 차이: 106620000


6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성  

온라인 쇼핑몰 데이터를 pandas로 분석하고 있습니다.
고객별 구매 금액을 계산하려고 하는데 다음과 같이 분석하려고 합니다.
주문 상세 금액은 `quantity × unit_price`로 계산하고 주문 상태가 `completed`인 주문만 분석에 포함하려고 합니다.
사용할 데이터는 다음과 같습니다.
- `order_items`
  - `order_id`, `quantity`, `unit_price`
- `orders`
  - `order_id`, `customer_id`, `order_status`
- `customers`
  - `customer_id`, `gender`, `age`, `city`
`order_items`와 `orders`는 `order_id`로 연결하고 고객별 집계 결과는 `customer_id`를 기준으로 `customers`와 연결하려고 합니다.
병합 과정에서는 잘못된 중복이나 누락을 확인하기 위해 `validate`와 `indicator`를 사용하고 병합 전후 행 수와 미매칭 데이터도 확인하고 싶습니다.
고객별 결과에는 다음 값을 포함해 주세요.
- `customer_id`
- `order_count`
- `total_sales`
- `city`
여기서 주문 수는 주문 상세 행 수가 아니라 `order_id.nunique()`를 이용한 실제 주문 건수로 계산해 주세요.
마지막으로 고객별 `total_sales` 합계가 전체 completed 주문의 `line_total` 합계와 일치하는지도 확인해 주세요.
위 기준에 맞는 pandas 코드와 각 검증 단계가 필요한 이유를 간단히 설명해 주세요.

# Chapter 04 제출 답안 양식. pandas로 데이터에 질문하기

> 주 제출물은 실행 완료 Notebook `chapter04/chapter04.ipynb`입니다. 이 양식의 항목을 Notebook의 Markdown 셀로 추가해 작성합니다.

## 0. 제출 정보
- 이름:김희철
- GitHub ID:KHeeCheol
- 작성일:2026-10-02
- 최종 제출 URL:https://github.com/KHeeCheol/llm-data-analysis-study/blob/main/notebooks/chapter04.ipynb
## 1. 질문과 필요한 데이터 선택
### 내가 확인하려는 질문
어떤 상품 카테고리의 completed 주문 기준 금액이 가장 큰가?

### 사용한 파일/컬럼
- `order_items`
  - `order_id`
  - `product_id`
  - `quantity`
  - `unit_price`

- `orders`
  - `order_id`
  - `order_status`

- `products`
  - `product_id`
  - `category`

### 결과 관찰
4개 CSV가 모두 정상적으로 로딩되었고 분석에 필요한 컬럼도 실제 데이터에 존재하였다.
분석에 필요한 `order_id`, `product_id`, `quantity`, `unit_price`,
`order_status`, `category` 컬럼이 실제 데이터에 존재하는 것을 확인하였다.

### 나의 해석과 판단
왜 이 컬럼과 범위를 선택했는지 작성하세요.
주문 금액은 `order_items`의 수량과 단가를 이용해 계산할 수 있지만 완료 주문만 선택하려면 `orders`의 주문 상태가 필요하다.
또한 상품 카테고리별로 비교하려면 `products`의 카테고리 정보가 필요하므로 세 데이터를 연결해야 한다고 판단하였다.

### 업무·분석적 의미
분석 전에 질문에 필요한 데이터와 컬럼을 먼저 정하면 불필요한 데이터를 사용하거나 실제로 존재하지 않는 컬럼을 기준으로 분석하는 오류를 줄일 수 있다.
### 한계와 추가 확인 사항
현재 단계에서는 필요한 컬럼의 존재 여부만 확인하였다. 실제 주문 상태의 값, 키의 관계, 누락이나 중복 여부 등은 이후 단계에서 추가로 확인해야 한다.

## 2. 필터링·정렬·파생 컬럼
- 적용한 필터 조건:
  - `age >= 30`
  - `city == "서울"`
  - `city`가 `"서울"` 또는 `"부산"`

- 정렬 기준:
  - 상품 가격 `price` 내림차순
  - 고객 나이 `age` 내림차순

- 만든 파생 컬럼:
  - `line_total`

- `line_total` 계산식:
  - `quantity × unit_price`

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰
필터링 결과 30세 이상 고객은 111명, 서울 고객은 15명, 서울 또는 부산 고객은 31명으로 확인되었다.
상품은 가격이 높은 순서로 고객은 나이가 높은 순서로 정상적으로 정렬되었다.
주문 상세 데이터에서 `quantity × unit_price`로 `line_total`을 계산했으며 전체 주문 상세 금액은 255,610,000으로 확인되었다.

### 나의 해석과 판단
필터 조건과 정렬 기준에 따라 분석 대상과 결과의 표시 순서가 달라질 수 있으므로 분석 목적에 맞는 조건을 명확하게 설정할 필요가 있다.
또한 현재 계산한 `line_total`은 주문 상태를 아직 연결하지 않은 주문 상세 금액이므로 전체 합계를 completed 주문 금액으로 해석해서는 안 된다고 판단하였다.

### 업무·분석적 의미
같은 데이터라도 어떤 조건으로 행을 선택하는지에 따라 분석 대상이 라질 수 있다. 또한 금액 분석에서는 계산식뿐 아니라 어떤 주문 상태를 포함하는지까지 함께 정의해야 지표의 의미를 명확하게 유지할 수 있다.

### 한계와 추가 확인 사항
현재 `line_total`은 `quantity × unit_price`만 반영한 값이며
아직 completed, cancelled, refunded 주문을 구분하지 않았다.
따라서 이후 주문 데이터를 병합한 뒤 completed 주문만 선택하여
이번 과제에서 사용하는 분석 범위를 확정해야 한다.

## 3. merge 검증
### 주문 상세와 주문 데이터 병합

- 병합한 데이터: `order_items`와 `orders`
- 사용한 key: `order_id`
- `validate`: `many_to_one`
- `indicator` 결과: `both` 764, `left_only` 0, `right_only` 0
- 병합 전/후 행 수: 764 → 764

### 완료 주문 상세와 상품 데이터 병합

- 병합한 데이터: `completed_order_sales`와 `products`
- 사용한 key: `product_id`
- `validate`: `many_to_one`
- `indicator` 결과: `both` 474, `left_only` 0, `right_only` 0
- 병합 전/후 행 수: 474 → 474

![merge 검증](images/step03_merge.png)

### 결과 관찰
`orders.order_id`의 중복 수는 0건이었다.
`order_items`와 `orders`를 `order_id` 기준으로 병합한 결과 병합 전과 후의 행 수가 모두 764행으로 동일하였다. 또한 `_merge` 결과는 764행 모두 `both`였으며 연결되지 않은 데이터는 없었다.

### 나의 해석과 판단
`orders.order_id`가 중복되지 않았고,`validate="many_to_one"` 조건으로 병합이 정상적으로 수행되었다.또한 병합 전후 행 수가 변하지 않았고 모든 행이 정상적으로 연결되었기 때문에 현재 데이터에서는 의도한 관계로 병합되었다고 판단하였다.


### 업무·분석적 의미
병합 과정에서 기준 key가 중복되어 행 수가 증가하면 같은 주문 상세 금액이 여러 번 집계되어 분석 결과가 실제보다 크게 계산될 수 있다.
따라서 병합 전 key의 중복 여부와 병합 후 행 수 및 미매칭 여부를
확인하는 과정이 필요하다.

### 한계와 추가 확인 사항
현재 확인한 내용은 key 관계와 병합 구조가 정상이라는 의미이다.
병합된 각 데이터의 값 자체가 업무적으로 정확한지까지 보장하는 것은 아니다.

## 4. completed 주문 범위와 집계
- 분석 범위 정의:
  - `order_status == "completed"`인 주문만 사용
  - `line_total = quantity × unit_price`
  - `total_sales`는 completed 주문의 `line_total` 합계

- 카테고리별 결과:
  - 스포츠: 31,743,000
  - 전자기기: 26,400,000
  - 생활용품: 23,915,000
  - 뷰티: 23,383,000
  - 식품: 16,573,000
  - 도서: 16,389,000
  - 패션: 10,587,000

- 상품별 결과:
  - 스포츠 상품 041: `total_sales` 5,705,000, `total_quantity` 35

- 월별 결과:
  - 2025-10: `total_sales` 25,766,000, `order_count` 26
  - 월별 금액은 기간 동안 일정하게 증가하거나 감소하지 않고 등락이 반복됨

- 고객별 결과:
  - Customer 117: `total_sales` 4,100,000, `order_count` 5

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰
카테고리별 `total_sales`를 비교한 결과 스포츠가 31,743,000으로 가장 높았고 패션이 10,587,000으로 가장 낮았다. 스포츠는 전체 카테고리 금액의 21.31%를 차지했다.
상품별로는 스포츠 상품 041의 `total_sales`가 5,705,000으로 가장 높았고
`total_quantity`는 35개였다.
월별로는 2025년 10월의 `total_sales`가 25,766,000으로 가장 높았다.
2025년 10월 이후 11월에는 감소했고, 12월과 2026년 1월에는 다시 증가했으며
이후에도 감소와 증가가 반복되어 전체 기간에서 일정한 상승 또는 하락 추세는 나타나지 않았다.
고객별로는 Customer 117의 `total_sales`가 4,100,000으로 가장 높았으며
완료 주문 건수는 5건이었다.


### 나의 해석과 판단
가장 중요하다고 판단한 결과는 스포츠 카테고리의 `total_sales`가 31,743,000으로 가장 높았다는 결과이다.
이번 분석의 핵심 질문이 "어떤 상품 카테고리의 completed 주문 기준 금액이 가장 큰가?"이므로 스포츠 카테고리가 가장 높다는 결과가 질문에 직접 답하는 결과이기 때문이다.
다만 스포츠의 `total_sales`가 가장 높다는 것만으로 스포츠 상품의 인기가 가장 높거나 수익성이 가장 좋다고 판단할 수는 없다.
현재 결과에서는 스포츠의 판매 수량도 295개로 가장 많았기 때문에 이후에는 높은 금액이 판매 수량의 영향인지 판매 단가의 영향인지 추가로 확인할 필요가 있다고 판단하였다.

### 업무·분석적 의미
다음 의사결정/분석으로 어떻게 연결할지 작성하세요.
이번 결과를 다음 분석으로 연결한다면 우선 스포츠 카테고리의 높은 `total_sales`가 판매 수량, 단가, 주문 건수 중 어떤 요인의 영향을 크게 받았는지 세부적으로 확인할 수 있다.
이후 이러한 분석 결과를 재고 운영이나 상품 구성, 프로모션 검토의 기초 자료로 사용할 수 있다.
다만 실제 의사결정을 하기 전에는 주문 건수, 고객 수, 원가와 수익성 등의 지표를 함께 검토할 필요가 있다.

### 한계와 추가 확인 사항
이번 `total_sales`는 `order_status == "completed"`인 주문에서
`quantity × unit_price`를 합산한 값이다. 할인, 쿠폰, 배송비, 세금, 원가, 마진, 부분 환불 등의 정보는 반영되어 있지 않기 때문에 이를 실제 회사의 회계상 순매출이나 이익이라고 단정할 수 없다.
또한 월별 결과에서 첫 달과 마지막 달이 한 달 전체 기간을 포함하는지 여부를 확인하지 않은 상태이므로 월별 금액 차이만으로 증가 또는 감소 추세를 단정하기에는 한계가 있다.

## 5. 총합 일치 검증

- 원본 completed `line_total` 합계: 148,990,000
- 카테고리 합계: 148,990,000
- 월별 합계: 148,990,000
- 고객별 합계: 148,990,000
- 차이 여부: 없음

추가로 상품별 합계도 148,990,000으로 동일하였다.

![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단
총합 검증이 필요한 이유와 불일치 시 확인할 순서를 작성하세요.
총합 검증이 필요한 이유는 동일한 데이터를 서로 다른 기준으로 집계하는 과정에서
필터 조건이 달라지거나 merge로 행이 중복 또는 누락되면 각 집계 결과의 금액이 달라질 수 있기 때문이다.
이번 결과에서는 모든 합계가 동일하므로 현재까지의 선택·병합·집계 과정에서 completed 주문 범위가 일관되게 유지된 것으로 판단하였다.

만약 총합이 일치하지 않는다면 다음 순서로 확인한다.
1. 각 분석에서 `completed` 필터가 동일하게 적용되었는지 확인한다.
2. `line_total = quantity × unit_price` 계산식이 동일한지 확인한다.
3. merge 전후 행 수가 증가하거나 감소하지 않았는지 확인한다.
4. merge에서 미매칭 데이터가 발생하지 않았는지 확인한다.
5. groupby 기준 컬럼에 결측값이 있어 일부 행이 제외되지 않았는지 확인한다.
6. 집계 과정에서 추가적인 필터가 적용되지 않았는지 확인한다.

## 6. LLM pandas 코드 검증
- LLM Prompt 요약:
  - Notebook에 제시된 카테고리별 주문 금액 집계 코드를 검토하고 `category` 컬럼의 실제 위치와 completed 주문 필터링 `merge`의 `validate`와 `indicator`를 포함한 안전한 수정 코드와 검증 순서를 요청.

- 제안 코드 요약:
  - 처음 제시된 LLM 코드는 다음과 같이
    `order_items`에서 바로 `category`를 기준으로 `line_total`을 합산하는 코드였다.

  `category_sales = order_items.groupby('category')['line_total'].sum()`

- 실제 컬럼/범위와 맞지 않은 부분:
  - `category` 컬럼은 `order_items`가 아니라 `products`에 존재하였다.
  - `order_status`는 `orders`에 있으므로 기존 코드에서는 `completed` 주문만 선택할 수 없었다.
  - `orders`와 `products`를 연결하기 위한 merge 과정이 없었다.
  - `validate`, `indicator`, 병합 전후 행 수와 미매칭 확인 등의 병합 검증 과정도 포함되어 있지 않았다.

- 수정한 내용:
  - `order_id`를 기준으로 `order_items`와 `orders`를 `many_to_one` 관계로 병합하였다.
  - `indicator`를 이용해 미매칭 여부와 병합 전후 행 수를 확인하였다.
  - `order_status == "completed"`인 주문만 분석 대상으로 선택하였다.
  - `product_id`를 기준으로 `products`를 다시 `many_to_one`으로 병합하여 `category`를 연결하였다.
  - 두 번째 병합도 `indicator`와 행 수를 이용해 검증하였다.
  - 마지막으로 `category`별 `line_total`을 집계하고 원본 completed 합계와 카테고리 합계가 일치하는지 확인하였다.

- 최종 판단: 수정 후 사용

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단
LLM 코드가 실행된다는 사실만으로 분석이 올바르다고 볼 수는 없다.
이번 실습에서도 실제 컬럼의 위치와 분석 범위를 확인하지 않으면 존재하지 않는 `category` 컬럼을 사용하거나 cancelled와 refunded 주문까지 포함한 결과를 만들 수 있었다. 또한 merge가 실행되더라도 key 중복이나 미매칭으로 인해
행이 증가하거나 누락되면 집계 금액이 달라질 수 있으므 `validate`, `indicator`, 병합 전후 행 수와 최종 합계를 직접 확인해야 한다.
따라서 LLM이 작성한 코드는 바로 사용하는 것이 아니라 실제 데이터 구조와 분석 조건을 기준으로 사람이 검증한 뒤 사용하는 것이 필요하다고 판단하였다.

## 7. Chapter 04 최종 인사이트
### 가장 의미 있다고 생각한 결과 2가지
1. completed 주문 기준 카테고리별 `total_sales`를 비교했을 때
   스포츠 카테고리가 가장 높은 금액을 기록하였다.

2. 카테고리별, 상품별, 월별, 고객별로 각각 집계한 `total_sales` 합계가
   모두 원본 completed 주문의 `line_total` 합계와 동일하게 나타났다.

### 그 결과를 뒷받침하는 수치/표
- 결과 1의 근거:
  - 스포츠 카테고리 `total_sales`: 31,743,000
  - 스포츠 카테고리 `total_quantity`: 295
  - 전체 카테고리 금액 중 비중: 21.31%
  - 카테고리 중 가장 높은 값으로 확인되었다.

- 결과 2의 근거:
  - 원본 completed `line_total` 합계: 148,990,000
  - 카테고리별 합계: 148,990,000
  - 상품별 합계: 148,990,000
  - 월별 합계: 148,990,000
  - 고객별 합계: 148,990,000
  - 전체 합계 일치 여부: `True`

### 추가로 확인하고 싶은 질문
스포츠 카테고리의 `total_sales`가 가장 높게 나타난 이유가 판매 수량이 많기 때문인지, 평균 판매 단가가 높기 때문인지 추가로 확인하고 싶다.
또한 월별 `total_sales`의 변화가 주문 건수 변화의 영향인지 평균 주문 금액 변화의 영향인지 비교해 보고 싶다.

### 현재 결과의 한계
이번 분석의 `total_sales`는 `order_status == "completed"`인 주문에서
`quantity × unit_price`를 합산한 값이다.
따라서 할인, 쿠폰, 배송비, 세금, 원가, 마진, 부분 환불 등의 정보는 반영되어 있지 않아 실제 회사의 회계상 순매출이나 이익으로 해석할 수 없다.
또한 카테고리별 금액 차이만으로 고객 선호나 프로모션 효과와 같은 원인을 판단할 수 없으며, 이를 확인하려면 추가적인 지표와 분석이 필요하다.

## 최종 제출 체크
- [O] 핵심 셀 Output이 남아 있습니다.
- [O] merge와 총합 검증 Evidence가 있습니다.
- [O] 결과 관찰과 해석이 구분되어 있습니다.
- [O] LLM 코드를 검증했습니다.
- [O] 개인정보/Secret이 없습니다.
- [O] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [O] 최종 Notebook 파일 URL을 제출합니다.

## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.
